# 실습 문제 모범답안 - 102. Memory Concepts

`102_Memory_Concepts` 노트북 실습 문제에 대한 모범답안 예시입니다.

### 구현 요약

1. `InMemorySaver` 에이전트로 thread 격리 확인 — 같은 thread는 기억, 다른 thread는 모름
2. `SqliteSaver`로 대화를 `my_memory.db`에 영구 저장
3. checkpointer를 새로 열어 같은 thread_id로 대화 복원 확인

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5.4-mini", model_provider="openai")

## 1. thread 격리 확인

checkpointer는 **thread_id별로** 대화를 저장합니다.
같은 thread에서는 이전 대화를 기억하지만, 다른 thread는 완전히 독립된 대화입니다.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model, tools=[], checkpointer=InMemorySaver())

config_a = {"configurable": {"thread_id": "user_a"}}
config_b = {"configurable": {"thread_id": "user_b"}}

# thread "user_a"에 정보 저장
result = agent.invoke(
    {"messages": [{"role": "user", "content": "내가 좋아하는 색은 파란색이야."}]},
    config_a,
)
print("user_a 응답:", result["messages"][-1].content)

# 같은 thread → 기억함
result = agent.invoke(
    {"messages": [{"role": "user", "content": "내가 좋아하는 색이 뭐라고 했지?"}]},
    config_a,
)
print("\n[같은 thread user_a]:", result["messages"][-1].content)

# 다른 thread → 기억하지 못함
result = agent.invoke(
    {"messages": [{"role": "user", "content": "내가 좋아하는 색이 뭐라고 했지?"}]},
    config_b,
)
print("\n[다른 thread user_b]:", result["messages"][-1].content)

## 2. SqliteSaver로 영구 저장

`my_memory.db` 파일에 대화가 저장되므로 프로세스가 재시작되어도 유지됩니다.

In [ ]:
import os
from langgraph.checkpoint.sqlite import SqliteSaver

DB_PATH = "my_memory.db"

# 이전 실행의 DB 파일 정리
for f in [DB_PATH, DB_PATH + "-wal", DB_PATH + "-shm"]:
    if os.path.exists(f):
        os.remove(f)

with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    checkpointer.setup()

    agent_persistent = create_agent(model, tools=[], checkpointer=checkpointer)
    config = {"configurable": {"thread_id": "food_thread"}}

    result = agent_persistent.invoke(
        {"messages": [{"role": "user", "content": "내가 좋아하는 음식은 김치찌개야."}]},
        config,
    )
    print("저장 완료:", result["messages"][-1].content)

## 3. 재로드 — DB에서 대화 복원

checkpointer를 **새로 열어도** 같은 `thread_id`를 쓰면 DB에 저장된 대화가 복원됩니다.

In [ ]:
with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
    agent_reload = create_agent(model, tools=[], checkpointer=checkpointer)

    # 같은 thread_id로 이전 대화 이어가기
    config = {"configurable": {"thread_id": "food_thread"}}

    result = agent_reload.invoke(
        {"messages": [{"role": "user", "content": "내가 좋아하는 음식이 뭐라고 했지?"}]},
        config,
    )
    print("재로드 후 응답:", result["messages"][-1].content)

## 확인 포인트

- 1번: 같은 thread(`user_a`)는 "파란색"을 기억하지만, 다른 thread(`user_b`)는 모릅니다 — **thread_id가 대화의 경계**입니다.
- 2~3번: checkpointer 객체를 새로 만들었는데도 답이 복원되는 이유는 대화가 메모리가 아니라 **DB 파일**(`my_memory.db`)에 있기 때문입니다.
  `InMemorySaver`였다면 새로 만든 순간 이전 대화가 사라집니다.